# Layer Normalization

$$\text{LayerNorm}(x) = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} \cdot \gamma + \beta$$

- $\mu$ — 当前样本的均值
- $\sigma^2$ — 当前样本的方差
- $\epsilon$ — 防止除零的小数（默认 1e-5）
- $\gamma, \beta$ — 可学习的缩放和平移参数，初始值为 1 和 0

## 为什么需要 LayerNorm？

神经网络每层的输出分布会随训练不断变化（Internal Covariate Shift），导致训练不稳定、收敛慢。LayerNorm 把每层输出强制归一化到均值为 0、方差为 1，再用 $\gamma, \beta$ 恢复表达能力。

## LayerNorm vs BatchNorm

| | LayerNorm | BatchNorm |
|---|---|---|
| 归一化维度 | 特征维度（单个样本内） | batch 维度（跨样本） |
| 适合场景 | NLP、Transformer | CV、CNN |
| batch size 依赖 | 无 | 有，batch 太小效果差 |

Transformer 用 LayerNorm 是因为 NLP 序列长度不固定，batch 内样本差异大，BatchNorm 效果不好。

In [1]:
import torch

In [2]:
def my_layer_norm(x, gamma, beta, eps=1e-5):
    mean = x.mean(dim=-1, keepdim=True)                    # 对特征维度求均值
    var = x.var(dim=-1, keepdim=True, unbiased=False)      # 有偏方差（除以 N，不是 N-1）
    x_norm = (x - mean) / torch.sqrt(var + eps)            # 归一化
    return gamma * x_norm + beta                           # 缩放和平移

## unbiased=False 是什么意思？

方差有两种计算方式：

$$\text{有偏（biased）：} \sigma^2 = \frac{1}{N}\sum(x_i - \mu)^2$$

$$\text{无偏（unbiased）：} \sigma^2 = \frac{1}{N-1}\sum(x_i - \mu)^2$$

统计学中无偏方差是对总体方差的无偏估计，但 LayerNorm 是对当前样本做归一化，不需要估计总体，所以用有偏方差（`unbiased=False`），和 PyTorch 官方实现保持一致。

## 验证

In [3]:
x = torch.randn(2, 8)      # batch=2，特征维度=8
gamma = torch.ones(8)      # 初始缩放为 1
beta = torch.zeros(8)      # 初始平移为 0

out = my_layer_norm(x, gamma, beta)
ref = torch.nn.functional.layer_norm(x, [8], gamma, beta)

print("Match ref?", torch.allclose(out, ref, atol=1e-4))

Match ref? True


In [6]:
x = torch.randn(2, 4)
print(x)

tensor([[ 1.6493,  0.3891, -1.1967, -1.3995],
        [ 0.7607,  1.0865,  0.1983,  1.1147]])


In [11]:
mean = x.mean(dim=-1, keepdim=True)
print(mean)
var = x.var(dim=-1, keepdim=True, unbiased=False)
print(var)
gamma = torch.ones(4)      # 初始缩放为 1
beta = torch.zeros(4)      # 初始平移为 0
eps=1e-5
x_norm = (x - mean) / torch.sqrt(var + eps)
print(x_norm)
out = gamma * x_norm + beta
print(out)


tensor([[-0.1394],
        [ 0.7901]])
tensor([[1.5462],
        [0.1361]])
tensor([[ 1.4386,  0.4251, -0.8503, -1.0134],
        [-0.0796,  0.8036, -1.6041,  0.8801]])
tensor([[ 1.4386,  0.4251, -0.8503, -1.0134],
        [-0.0796,  0.8036, -1.6041,  0.8801]])
